# Capstone: How to ship a RAG service with its release gate

**CPU | API key: `OPENAI_API_KEY` (chat and embedding calls; the release gate needs none) | Self-study**

## Goal

Put the pipeline you built behind a small web page that answers with citations and shows its sources and its trace,
and ship it only when Lab 11's release gate passes on the page's own results. The moment that teaches it: the first
page asked visitors who they are. A customer who answered *agent* read the support runbook's *4 hours* in 3 of 3
requests, through a permission filter that does exactly what Lab 11 tested, and hiding the field from the screen
changed nothing. The fix is a sign-in: the page, not the visitor, decides who is asking.

## What you will learn

- How to build and test the service's engine before any page: a permission filter, a cited answer, a quote check
  and one trace record per request.
- How to put the engine behind a Gradio page that shows its progress and shows the answer only after the quote check has run.
- How to test a page from code as several visitors with `gradio_client`, update by update.
- Why a page must take the user from a sign-in and never from a field the visitor fills, and how Gradio's `auth` does it.
- How to run the service from one file, `app.py`, on your route.
- How to run Lab 11's release gate on the service's own results, and in a CI job with no key.

## Set up the notebook

Run these two cells first, every time; Lab 00 explains each line. **Setup** finds the repository folder (on Colab it
copies the repository and installs the packages first). **Config** reads the settings and finds your key.

In [1]:
# Setup: on Colab, copy the repository and install the packages. Everywhere else they are installed already.
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not Path("/content/rag_for_nlp").exists():
        !git clone -q --depth 1 https://github.com/tatwan/rag_for_nlp.git /content/rag_for_nlp
    %cd /content/rag_for_nlp
    %pip install -q -r requirements.txt

ROOT = next(folder for folder in [Path.cwd(), *Path.cwd().parents] if (folder / "models.env").exists())
sys.path.insert(0, str(ROOT))
print("Repository folder:", ROOT.name)

Repository folder: rag_for_nlp


In [2]:
# Config: settings from models.env (your .env and environment variables win); the key from the environment, .env or Colab Secrets.
from shared.settings import load_settings, load_key, stamp

settings = load_settings()
key_found_in = load_key("OPENAI_API_KEY")

for name, value in stamp(settings).items():
    print(f"{name:18}{value}")
print(f"{'OPENAI_API_KEY':18}{'found in ' + key_found_in if key_found_in else 'not found'}")

chat_model        gpt-4o-mini
reasoning_effort  None
embed_model       text-embedding-3-small
judge_model       gpt-4o-mini
rerank_model      gpt-4o-mini
chat_endpoint     https://api.openai.com/v1
embed_endpoint    https://api.openai.com/v1
run_date          2026-10-19
OPENAI_API_KEY    found in .env


This lab adds [Gradio](https://www.gradio.app/docs), the library that draws the page, and `gradio_client`, which
opens a page from code. They are not in `requirements.txt`: with uv, install them with `uv sync --group capstone`;
with pip, `pip install "gradio>=6,<7"`. On Colab this cell installs them.

In [3]:
# Gradio draws the page; gradio_client opens it from code, as a visitor's browser does.
import warnings
warnings.filterwarnings("ignore", message="IProgress not found")   # a progress-bar notice from a Gradio dependency
if IN_COLAB:
    %pip install -q "gradio>=6,<7"
import gradio as gr
from gradio_client import Client

print("gradio", gr.__version__)

gradio 6.30.0


This lab uses `chat_model`, `embed_model` and `JSON_MODE`. It embeds the corpus three times and makes one short chat
call for each question it answers, 42 of them in Part E. The saved outputs and every count in the text come from `gpt-4o-mini`; the README
lists what `gpt-6-luna` did.

The service you will build, and the gate that checks it before it ships:

![One request through the service, and the release gate on its saved results](resources/service_flow.png)

---

## Part A: Build the engine and test it before any page

Part A writes the function the page will call and tests it in the notebook, where every output can be read.

The index is Lab 11's: the 272 chunks and the five policies labelled `public`, and four support records for agents
only, labelled `staff` and `internal`. `build_store()` does Lab 11's Part A in one function:

In [4]:
import json
from collections import Counter
from shared import lc_pipeline

CONSTRUCTED = json.loads((ROOT / "11_Release_Gate" / "resources" / "constructed-records.json").read_text(encoding="utf-8"))
USERS = CONSTRUCTED["users"]                             # which access labels each kind of user may read

def build_store():
    """Every record with its access label (public unless it has one), embedded fresh."""
    records = [{"access": "public", **record} for record in lc_pipeline.load_corpus()] + CONSTRUCTED["restricted"]
    return lc_pipeline.build_store(lc_pipeline.to_documents(records), settings)

store = build_store()
print(len(store.store), "documents |", Counter(doc["metadata"]["access"] for doc in store.store.values()))
print(USERS)

281 documents | Counter({'public': 277, 'staff': 3, 'internal': 1})
{'customer': ['public'], 'agent': ['public', 'staff', 'internal']}


281 documents: 277 `public`, 3 `staff`, 1 `internal`. A customer may read `public` only, an agent all three. The
permission filter is Lab 11's allow-list, which the store applies before it ranks anything. Search for the runbook
question as each user:

In [5]:
def can_read(user):
    """A filter for one user: True for a document whose access label the user may read."""
    return lambda document: document.metadata["access"] in USERS[user]

STAFF_QUESTION = "How long can a support agent make an Atlas reset link last?"
for user in USERS:
    print(f"{user:8}", [document.id for document, _ in lc_pipeline.retrieve(store, STAFF_QUESTION, filter=can_read(user))])

customer ['H02', 'H01', 'H04']


agent    ['R04', 'R01', 'H02']


As in Lab 11, the customer gets three public policies and the agent gets the incident note (R04), the runbook page
(R01) and H02. Answer the customer's question from the customer's hits, and count what the call costs:
[`get_usage_metadata_callback()`](https://reference.langchain.com/python/langchain-core/callbacks/usage/get_usage_metadata_callback)
collects the tokens of every chat model call made inside its `with` block, including calls inside a `shared/` helper.

In [6]:
import time
from langchain_core.callbacks import get_usage_metadata_callback

hits = lc_pipeline.retrieve(store, STAFF_QUESTION, filter=can_read("customer"))
lines = lc_pipeline.evidence(hits)
start = time.perf_counter()
with get_usage_metadata_callback() as usage:
    reply = lc_pipeline.answer(STAFF_QUESTION, lines, settings)
print(reply.answer, f"| {time.perf_counter() - start:.2f} s")
print(usage.usage_metadata)

A support agent can make an Atlas reset link last for 30 minutes. | 1.16 s
{'gpt-4o-mini-2024-07-18': {'input_tokens': 299, 'output_tokens': 51, 'total_tokens': 350, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}}


The customer's answer is *A support agent can make an Atlas reset link last for 30 minutes.* Its evidence is H02, which is about every reset
link, so the answer is faithful to what the customer may read, as in Lab 11. The tokens are listed under the model's
full name: 299 input and 51 output tokens.

Before the page shows an answer, it checks it: every quote must be in the evidence line it cites (Lab 03's
`quote_found()`), or the answer must be the refusal. `all([])` is `True`, so a plain `all(...)` would pass an answer
with no claims at all; `bool(claims)` stops that:

In [7]:
from shared.answer import quote_found
from shared.evals import REFUSAL

claims = [claim.model_dump() for claim in reply.claims]
refused = reply.answer.strip() == REFUSAL
checked = refused or (bool(claims) and all(quote_found(claim, lines) for claim in claims))
print("claims:", claims)
print("checked:", checked, "| an answer with no claims: all([]) is", all([]), "but the check gives", bool([]) and all([]))

claims: [{'claim': 'Atlas reset links expire after 30 minutes.', 'source_id': 'H02', 'quote': 'Atlas reset links expire after 30 minutes.'}]
checked: True | an answer with no claims: all([]) is True but the check gives False


The check passes. A **trace record** keeps what one request did, so its answer can be read again later (Lab 04): Lab
04's fields (hits with scores, packed IDs, a prompt hash, the reply, the tokens, the stamp) plus who asked, the
evidence lines, whether the check passed, and the seconds. With k=3 and no token budget, the packed IDs are the hits.
Each record is one JSON line in `.cache/capstone/traces.jsonl`, which git ignores because traces hold what users
asked. The cell starts the file empty, so the counts in this notebook are this run's:

In [8]:
import hashlib
from datetime import datetime
from shared.answer import RULES
from shared.settings import stamp

TRACES = ROOT / ".cache" / "capstone" / "traces.jsonl"   # gitignored: traces hold what users asked
TRACES.parent.mkdir(parents=True, exist_ok=True)
TRACES.write_text("", encoding="utf-8")                     # start empty: the counts below are this run's
print("trace file:", TRACES.relative_to(ROOT))

trace file: .cache/capstone/traces.jsonl


Each step has run on its own; combine them into the function the page will call. It answers, checks,
writes one trace line and returns the record:

In [9]:
def answer_and_trace(question, user, hits):
    """Answer from the hits, check the quotes, and append the request's trace record to TRACES."""
    lines = lc_pipeline.evidence(hits)
    start = time.perf_counter()
    with get_usage_metadata_callback() as usage:
        reply = lc_pipeline.answer(question, lines, settings)
    claims = [claim.model_dump() for claim in reply.claims]
    refused = reply.answer.strip() == REFUSAL
    tokens = usage.usage_metadata.values()
    record = {
        "time": datetime.now().isoformat(timespec="seconds"), "user": user, "question": question,
        "hits": [[document.id, round(score, 4)] for document, score in hits],
        "packed": [document.id for document, _ in hits], "evidence": lines,
        "prompt_sha": hashlib.sha256((RULES + question + "".join(lines)).encode()).hexdigest()[:12],
        "reply": {"answer": reply.answer, "claims": claims},
        "checked": refused or (bool(claims) and all(quote_found(claim, lines) for claim in claims)),
        "usage": {"input": sum(t["input_tokens"] for t in tokens), "output": sum(t["output_tokens"] for t in tokens)},
        "seconds": round(time.perf_counter() - start, 2), "stamp": stamp(settings),
    }
    TRACES.parent.mkdir(parents=True, exist_ok=True)
    with TRACES.open("a", encoding="utf-8") as file:
        file.write(json.dumps(record, ensure_ascii=False) + "\n")
    return record

Run it for the agent, and read the record it wrote:

In [10]:
record = answer_and_trace(STAFF_QUESTION, "agent", lc_pipeline.retrieve(store, STAFF_QUESTION, filter=can_read("agent")))
for name, value in record.items():
    print(f"{name:11}", value)
print(len(TRACES.read_text(encoding="utf-8").splitlines()), "line in", TRACES.relative_to(ROOT))

time        2026-10-19T00:53:43
user        agent
question    How long can a support agent make an Atlas reset link last?
hits        [['R04', 0.7208], ['R01', 0.7151], ['H02', 0.64]]
packed      ['R04', 'R01', 'H02']
evidence    ['[R04] When a customer says an Atlas reset link expired too soon, check the request time in the admin console before you send a new link.', '[R01] Support agents can send a reset link that stays valid for 4 hours to an Atlas workspace owner who is locked out. Do not offer it to other users.', '[H02] Atlas reset links expire after 30 minutes. Request a new link if the old link has expired.']
prompt_sha  2c9527ce8b78
reply       {'answer': 'Support agents can make an Atlas reset link last for 4 hours.', 'claims': [{'claim': 'Support agents can send a reset link that stays valid for 4 hours to an Atlas workspace owner who is locked out.', 'source_id': 'R01', 'quote': 'Support agents can send a reset link that stays valid for 4 hours to an Atlas workspace owner w

The agent's answer is *Support agents can make an Atlas reset link last for 4 hours.*, from R01, and `checked` is `true`. The record says who asked (`agent`), what was
searched and packed, and what it cost: 324 input and 80 output tokens, 1.42 s. Every request the service answers from now on writes
one such line.

### Checkpoint A

| You should see | Our run |
|---|---|
| The index | 281 documents: 277 `public`, 3 `staff`, 1 `internal` |
| The runbook question searched | customer: H02, H01, H04; agent: R04, R01, H02 |
| The customer's answer | *A support agent can make an Atlas reset link last for 30 minutes.* `checked: True` |
| The agent's trace record | `user: agent`, packed R04, R01, H02, *4 hours*, `checked: true`; 1 line in the trace file |

---

## Part B: Put the engine behind a page that streams its progress

Part B gives the engine a page. This first page asks for the question and for who the visitor is, because the engine
needs both.

A Gradio page function streams by `yield`: each `yield` replaces what the page shows. A reply streamed word by word
would reach the screen before its quotes are checked, and the check needs the whole reply. So the page streams its
progress (searching, then the sources it found) and shows the answer once, after the check. Two small functions turn
the hits and the trace record into the page's text. An answer that fails the check is still shown, under `WARNING`,
because this check also fails right answers whose quote the model shortened (Task 2 of the challenge shows one):

In [11]:
WARNING = "**Check the sources:** a quote below is not word for word in the line it cites."

def sources_text(hits):
    """The hits as the sources panel shows them: ID, title, access label, score and text."""
    return "\n\n".join(f"**[{document.id}]** {document.metadata.get('title', '')} · {document.metadata['access']} · "
                       f"{score:.3f}\n\n{document.page_content}" for document, score in hits)

def answer_text(record):
    """The answer with its quotes, and WARNING above it when the quote check failed."""
    quotes = [f"- [{claim['source_id']}] *{claim['quote']}*" for claim in record["reply"]["claims"]]
    return "\n\n".join(([] if record["checked"] else [WARNING]) + [record["reply"]["answer"], *quotes])

`respond_as()` is the page's function. Each `yield` gives the page's three outputs: the answer box, the sources panel
and the trace.

In [12]:
def respond_as(question, user):
    """The first page's function: the visitor says who they are."""
    yield "*Searching...*", "", None
    hits = lc_pipeline.retrieve(store, question, filter=can_read(user))
    yield f"*Answering from {len(hits)} sources...*", sources_text(hits), None
    record = answer_and_trace(question, user, hits)
    yield answer_text(record), sources_text(hits), record

[`gr.Interface`](https://www.gradio.app/docs/gradio/interface) turns a function into a page: one input component per
argument, one output component per value it yields; the radio button is the second argument, `user`. `LAUNCH` runs
the server in the background, keeps it on your machine (`share=False`) and shows it under the cell on Colab only.

In [13]:
LAUNCH = dict(prevent_thread_lock=True, share=False, inline=IN_COLAB, quiet=True)

def outputs():
    return [gr.Markdown(label="Answer"), gr.Markdown(label="Sources"), gr.JSON(label="Trace")]

choose_user = gr.Radio(list(USERS), value="customer", label="I am a")
page = gr.Interface(respond_as, [gr.Textbox(label="Question"), choose_user], outputs(),
                    title="Ask with sources", flagging_mode="never")
page.launch(**LAUNCH)
print("the page is at", page.local_url)

the page is at http://127.0.0.1:7860/


Open the address in a browser and ask a question. From code,
[`gradio_client`](https://www.gradio.app/guides/getting-started-with-the-python-client) opens the page as a browser
does: each `Client` is one visitor, and `submit()` yields every update the page shows, in order. `watch()` records
each update and when it arrived. Ask a public question as a customer:

In [14]:
def watch(visitor, *inputs, api_name):
    """Every update the page shows for one request: seconds since the request, answer, sources, trace."""
    start, updates = time.perf_counter(), []
    for answer, sources, trace in visitor.submit(*inputs, api_name=api_name):
        updates.append((round(time.perf_counter() - start, 2), answer, sources, trace))
    return updates

visitor = Client(page.local_url, verbose=False)
for seconds, answer, sources, trace in watch(visitor, "How long does an Atlas reset link remain valid?", "customer",
                                             api_name="/respond_as"):
    print(f"{seconds:5} s | sources: {sources.count('**[')} | trace: {trace is not None:1} | {answer!r}")

 0.08 s | sources: 0 | trace: 0 | '*Searching...*'
 0.24 s | sources: 3 | trace: 0 | '*Answering from 3 sources...*'
 1.33 s | sources: 3 | trace: 1 | 'Atlas reset links expire after 30 minutes.\n\n- [H02] *Atlas reset links expire after 30 minutes.*'


Three updates. The first two show progress; the sources panel fills in the second, 0.24 s after the
question. The answer and its trace arrive together in the third, after 1.33 s, once the quote check has run.
No update shows an answer the check has not seen. The API name, `/respond_as`, is the function's name.

### Checkpoint B

| You should see | Our run |
|---|---|
| The page | running at `http://127.0.0.1:7860/` (another port if 7860 is busy) |
| Updates | 3: *Searching...*, then three sources, then the answer with its quote and the trace |
| The answer | *Atlas reset links expire after 30 minutes.* |

---

## Part C: Decide who is asking

Part C tests the page as a visitor who is a customer, and finds out who the page lets that visitor be.

The radio button says who the visitor is, and the visitor chooses it. Predict what a customer gets who picks *agent*.
The customer asks the runbook question once as a customer, then three times as an agent:

In [15]:
customer = Client(page.local_url, verbose=False)
for user in ["customer", "agent", "agent", "agent"]:
    answer, sources, trace = customer.predict(STAFF_QUESTION, user, api_name="/respond_as")
    print(f"says {user:8} | trace user: {trace['user']:8} | {trace['packed']} | {trace['reply']['answer']}")

says customer | trace user: customer | ['H02', 'H01', 'H04'] | A support agent can make an Atlas reset link last for 30 minutes.


says agent    | trace user: agent    | ['R04', 'R01', 'H02'] | Support agents can make an Atlas reset link last for 4 hours.


says agent    | trace user: agent    | ['R04', 'R01', 'H02'] | Support agents can make an Atlas reset link last for 4 hours.


says agent    | trace user: agent    | ['R04', 'R01', 'H02'] | Support agents can make an Atlas reset link last for 4 hours.


As a customer: H02, H01, H04 and *30 minutes*. As an agent, three times: R04, R01 and H02, and *Support agents can make an Atlas reset link last for 4 hours.*
The permission filter did its job for the user it was given, and it was given the visitor's word. The trace says
`agent` too, so anyone reading the log later sees an agent asking.

The first fix that comes to mind is to hide the field. `visible=False` takes the radio button off the screen. Look at
the page's API, then send *agent* anyway:

In [16]:
page.close()
hidden_user = gr.Radio(list(USERS), value="customer", visible=False)
page = gr.Interface(respond_as, [gr.Textbox(label="Question"), hidden_user], outputs(),
                    title="Ask with sources", flagging_mode="never")
page.launch(**LAUNCH)
customer = Client(page.local_url, verbose=False)
api = customer.view_api(print_info=False, return_format="dict")["named_endpoints"]["/respond_as"]
print("parameters:", [parameter["parameter_name"] for parameter in api["parameters"]])
for _ in range(3):
    answer, sources, trace = customer.predict(STAFF_QUESTION, "agent", api_name="/respond_as")
    print(trace["packed"], trace["reply"]["answer"])
page.close()

Closing server running on port: 7860


parameters: ['question', 'user']


['R04', 'R01', 'H02'] Support agents can make an Atlas reset link last for 4 hours.


['R04', 'R01', 'H02'] Support agents can make an Atlas reset link last for 4 hours.


['R04', 'R01', 'H02'] Support agents can make an Atlas reset link last for 4 hours.
Closing server running on port: 7860


The screen now shows one box, the API still takes two parameters, and the customer read R01 in 3 of 3. Hiding a field
hides it from the screen, not from the API: anything the browser sends, a visitor can send from code.

So the page has to learn who is asking from something the visitor cannot choose: a **sign-in**. With
`launch(auth=...)`, Gradio asks for a login and password first
([Gradio: password-protected app](https://www.gradio.app/guides/sharing-your-app#password-protected-app)) and gives the
function a [`gr.Request`](https://www.gradio.app/docs/gradio/request) whose `username` is the login. `SIGN_INS` maps
each login to a kind of user, on the server. The passwords are made fresh on each run and never printed:

In [17]:
import secrets

SIGN_INS = {"casey": "customer", "alex": "agent"}       # demo sign-ins; a real service uses your company's sign-in
PASSWORDS = {login: secrets.token_urlsafe(8) for login in SIGN_INS}

def respond(question, request: gr.Request):
    """The page's function: the user comes from the sign-in, never from the visitor's input."""
    user = SIGN_INS[request.username]
    yield "*Searching...*", "", None
    hits = lc_pipeline.retrieve(store, question, filter=can_read(user))
    yield f"*Answering from {len(hits)} sources...*", sources_text(hits), None
    record = answer_and_trace(question, user, hits)
    yield answer_text(record), sources_text(hits), record

`respond()` has no `user` argument, so the page has no field to fill. Start it with the sign-in and send three
visitors: casey (a customer), alex (an agent), and one who does not sign in. Casey also tries to send *agent* as a
second value:

In [18]:
page = gr.Interface(respond, gr.Textbox(label="Question"), outputs(), title="Ask with sources", flagging_mode="never")
page.launch(auth=list(PASSWORDS.items()), **LAUNCH)
visitors = {login: Client(page.local_url, verbose=False, auth=(login, PASSWORDS[login])) for login in SIGN_INS}
for login, visitor in visitors.items():
    for _ in range(3):
        answer, sources, trace = visitor.predict(STAFF_QUESTION, api_name="/respond")
        print(f"{login:5} | trace user: {trace['user']:8} | {trace['packed']} | {trace['reply']['answer']}")
answer, sources, trace = visitors["casey"].predict(STAFF_QUESTION, "agent", api_name="/respond")
print("casey sends agent too | trace user:", trace["user"], "|", trace["packed"])
try:
    Client(page.local_url, verbose=False)
except Exception as error:
    print("no sign-in |", type(error).__name__, "|", error)
page.close()

casey | trace user: customer | ['H02', 'H01', 'H04'] | A support agent can make an Atlas reset link last for 30 minutes.


casey | trace user: customer | ['H02', 'H01', 'H04'] | A support agent can make an Atlas reset link last for 30 minutes.


casey | trace user: customer | ['H02', 'H01', 'H04'] | A support agent can make an Atlas reset link last for 30 minutes.


alex  | trace user: agent    | ['R04', 'R01', 'H02'] | Support agents can make an Atlas reset link last for 4 hours.


alex  | trace user: agent    | ['R04', 'R01', 'H02'] | Support agents can make an Atlas reset link last for 4 hours.


alex  | trace user: agent    | ['R04', 'R01', 'H02'] | A support agent can make an Atlas reset link last for 4 hours.


casey sends agent too | trace user: customer | ['H02', 'H01', 'H04']
no sign-in | AuthenticationError | Could not load http://127.0.0.1:7860/ as credentials were not provided. Please login.


Closing server running on port: 7860


Casey gets the public policies and *30 minutes* in 3 of 3; alex gets the runbook and *4 hours* in 3 of 3. The extra
*agent* that casey sent was dropped, because the page has one parameter, and the visitor who did not sign in got no
page at all. The filter is the same function as before; what changed is where its user comes from. Decide the user
at the boundary, where the request enters, and test the filter through the page: in the notebook it passed every test
while the page let a customer past it.

In a browser, the page shows a sign-in form first; to switch user, open the page's `/logout` address. A real service
takes the user from your company's sign-in (single sign-on) the same way: through the request, never from the form.

### Checkpoint C

| You should see | Our run |
|---|---|
| A customer who says *agent* | R04, R01, H02 and *4 hours*, 3 of 3; the trace says `agent` |
| The field hidden | `parameters: ['question', 'user']`; R04, R01, H02, 3 of 3 |
| Signed in | casey: H02, H01, H04, *30 minutes*, 3 of 3; alex: R04, R01, H02, *4 hours*, 3 of 3 |
| Casey sends *agent*, and no sign-in | `trace user: customer`; `AuthenticationError` |

---

## Part D: Run the service from `app.py`

Part D puts the engine and the signed-in page in one file, which is how the service runs outside the notebook.

`app.py` in this folder holds the functions of Parts A to C and adds three things: it stops with a message when no
key is found, it builds its index when it starts (`store = build_store()`), and when you run it as `python app.py` it
makes a password for each sign-in, prints them and launches the page with `auth`. Importing it builds the service's
own index. Read it:

In [19]:
import inspect

sys.path.insert(0, str(ROOT / "Capstone_RAG_Service"))
import app

print(inspect.getsource(app))

"""The RAG service: the pipeline of Labs 05 and 11 behind a sign-in page (Gradio).

The capstone notebook builds and tests each function here first. Run it from the repository folder:

    python Capstone_RAG_Service/app.py

It prints a new password for each demo sign-in, then serves the page at http://127.0.0.1:7860. Every request
appends one trace record to .cache/capstone/traces.jsonl (gitignored).
"""
import hashlib
import json
import secrets
import sys
import time
from datetime import datetime
from pathlib import Path

import gradio as gr
from langchain_core.callbacks import get_usage_metadata_callback

ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(ROOT))             # so that "shared" is found from any folder

from shared import lc_pipeline  # noqa: E402
from shared.answer import RULES, quote_found  # noqa: E402
from shared.evals import REFUSAL  # noqa: E402
from shared.settings import load_key, load_settings, stamp  # noqa: E402

settings = load_settings()
if

Check that the file holds the functions you tested in Parts A to C:

In [20]:
for name in ["build_store", "can_read", "answer_and_trace", "sources_text", "answer_text", "respond"]:
    print(f"{name:17} same as in the notebook: {inspect.getsource(getattr(app, name)) == inspect.getsource(globals()[name])}")

build_store       same as in the notebook: True
can_read          same as in the notebook: True
answer_and_trace  same as in the notebook: True
sources_text      same as in the notebook: True
answer_text       same as in the notebook: True
respond           same as in the notebook: True


All six are the same, so the tests above hold for the file. To run the service yourself, open a terminal in the
repository folder and start it:

```bash
uv run python Capstone_RAG_Service/app.py        # uv route (after uv sync --group capstone)
python Capstone_RAG_Service/app.py               # pip route, with .venv active
```

It prints one password for casey and one for alex, then `Running on local URL: http://127.0.0.1:7860`. Open that
address, sign in as casey, ask the runbook question, then open `/logout` and sign in as alex. Stop it with `Ctrl+C`.
The passwords change at every start. The README says how to open the page in a codespace, on a VM and on Colab.

### Checkpoint D

| You should see | Our run |
|---|---|
| The file | the functions of Parts A to C, then `store`, `build_page` and the launch |
| The check | six functions, each `same as in the notebook: True` |

---

## Part E: Run the release gate on the service's results

Part E asks the service Lab 11's release questions through its page, saves what the page returned as a results file,
and runs Lab 11's gate on it, unchanged.

Start the service's page, built from `app.py` with its own index, and sign in one visitor for each kind of user:

In [21]:
service = app.build_page()
service.launch(auth=list(PASSWORDS.items()), **LAUNCH)
login_for = {user: login for login, user in app.SIGN_INS.items()}
visitors = {user: Client(service.local_url, verbose=False, auth=(login, PASSWORDS[login])) for user, login in login_for.items()}
print("the service is at", service.local_url, "| signed in:", login_for)

the service is at http://127.0.0.1:7860/ | signed in: {'customer': 'casey', 'agent': 'alex'}


The release set is Lab 11's 42 questions: the 38 golden questions asked by a customer, three agent questions and one
customer question that must be refused. `row()` asks one question through the page as the record's user, and builds
the row Lab 11's tests read from the trace the page returned: the user the service decided, the packed IDs, the
answer, its claims and its evidence. `passed` is Lab 06's check (the reference found, or the refusal when one is
due); an answer shown under the warning counts as answered, because the visitor saw it:

In [22]:
from shared import evals

release_set = [{**record, "user": "customer"} for record in evals.golden_set()] + CONSTRUCTED["cases"]

def row(record):
    """One release question through the page, as the record's user; the row comes from the page's trace."""
    *_, trace = visitors[record["user"]].predict(record["question"], api_name="/respond")
    answer = trace["reply"]["answer"]
    refused = answer.strip() == REFUSAL
    return {"question_id": record["question_id"], "user": trace["user"], "type": record["type"],
            "refuse": record["refuse"], "packed": trace["packed"], "answer": answer, "refused": refused,
            "passed": refused if record["refuse"] else evals.found(answer, record["references"]),
            "claims": trace["reply"]["claims"], "evidence": trace["evidence"], "checked": trace["checked"]}

start = time.perf_counter()
rows = [row(record) for record in release_set]
print(f"{len(rows)} questions in {time.perf_counter() - start:.0f} s | passed {sum(r['passed'] for r in rows)}",
      f"| shown with the warning {sum(not r['checked'] for r in rows)}")
for r in rows:
    if not r["checked"]:
        print("with the warning |", r["question_id"], "|", r["answer"], "|", [claim["quote"] for claim in r["claims"]])

42 questions in 61 s | passed 38 | shown with the warning 0


38 of 42 passed, and 0 carried the warning. An answer that fails the quote check is still shown, under the warning; the README lists our other runs.

The gate also needs Lab 11's build manifest of the index the service searched, `app.store`. Save it with the rows as
`results/service.json`:

In [23]:
def sha(text):
    return hashlib.sha256(text.encode("utf-8")).hexdigest()[:12]

documents = [{"id": doc["id"], "source_id": doc["metadata"]["source_id"], "version": doc["metadata"].get("version"),
              "access": doc["metadata"]["access"], "text": sha(doc["text"])} for doc in app.store.store.values()]
manifest = {"search": "sign-in, allow-list per user, k=3", "k": 3, "rules": sha(RULES), **stamp(settings),
            "documents": documents}
RESULTS = ROOT / "Capstone_RAG_Service" / "results"
RESULTS.mkdir(exist_ok=True)
(RESULTS / "service.json").write_text(json.dumps({"manifest": manifest, "rows": rows}, indent=1, ensure_ascii=False),
                                      encoding="utf-8")
print(len(documents), "documents in the manifest | saved", (RESULTS / "service.json").relative_to(ROOT))

281 documents in the manifest | saved Capstone_RAG_Service/results/service.json


Run the gate as Lab 11 did: in a new Python process, from the repository folder, with the key set to empty.
`GATE_BASELINE` names Lab 11's saved results for today's release, `GATE_CANDIDATE` the service's file:

In [24]:
import os
import subprocess

env = {**os.environ, "OPENAI_API_KEY": "", "GATE_BASELINE": "11_Release_Gate/results/baseline.json",
       "GATE_CANDIDATE": "Capstone_RAG_Service/results/service.json"}
command = [sys.executable, "-m", "pytest", "11_Release_Gate/test_release.py", "-q", "--tb=short", "--color=no"]
print(subprocess.run(command, cwd=ROOT, env=env, capture_output=True, text=True).stdout)

.....                                                                    [100%]
5 passed in 0.02s



`5 passed in 0.02s`. No pack holds a document its user may not read, the index holds only current versions, the refusals hold, no question that today's release passed fails now, and the quotes are found. Compare the service with today's
release question by question, with Lab 08's `paired()`:

In [25]:
from shared.hybrid import paired

baseline = json.loads((ROOT / "11_Release_Gate" / "results" / "baseline.json").read_text(encoding="utf-8"))["rows"]
change = paired([r["passed"] for r in baseline], [r["passed"] for r in rows])
print("gained:", [baseline[i]["question_id"] for i in change["gained"]], "| lost:", [baseline[i]["question_id"] for i in change["lost"]])
print("failed:", [(r["question_id"], r["answer"][:45]) for r in rows if not r["passed"]])
service.close()

gained: ['A01', 'A02', 'A03'] | lost: []
failed: [('56e16839cd28a01900c67887', 'Computational complexity theory broadly defin'), ('G05', 'No, temporary Atlas accounts cannot use the u'), ('G08', 'Only workspace owners can export all reports '), ('G09', 'Insufficient evidence.')]
Closing server running on port: 7860


The service gains the three agent questions, A01, A02 and A03, and loses none: every question it fails, today's
release fails too.

The gate reads only saved files, so a CI job can run it on every change with no key. `release-gate.yml` in this
folder is a [GitHub Actions](https://docs.github.com/en/actions/writing-workflows/quickstart) workflow that does:

In [26]:
print((ROOT / "Capstone_RAG_Service" / "release-gate.yml").read_text(encoding="utf-8"))

# Release gate: Lab 11's release tests on the service's saved results. No model call, no API key.
# To turn it on, copy this file to .github/workflows/release-gate.yml at the top of your repository.
name: Release gate

on:
  pull_request:
  push:
    branches: [main]

jobs:
  gate:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v7
      - uses: actions/setup-python@v7
        with:
          python-version: "3.12"
      - run: pip install -r requirements.txt
      - run: python -m pytest 11_Release_Gate/test_release.py -q
        env:
          GATE_BASELINE: 11_Release_Gate/results/baseline.json
          GATE_CANDIDATE: Capstone_RAG_Service/results/service.json



On every pull request and push to `main` it installs `requirements.txt` and runs the same pytest command. To switch it
on, copy the file to `.github/workflows/` at the top of your repository. Whoever changes the service runs Part E again
and commits the new `results/service.json` with the change; a failing test then fails the pull request. The job
cannot tell whether the results came from the code in the pull request, so review the results file with the code.

### Checkpoint E

| You should see | Our run |
|---|---|
| The release set through the page | 42 questions, passed 38, shown with the warning 0 |
| Saved | `Capstone_RAG_Service/results/service.json`, 281 documents in the manifest |
| The gate | `5 passed in 0.02s` |
| Paired with today's release | gained A01, A02 and A03; lost none |

---

## What you can do now

- Build a service's engine and test it before any page: a permission filter, a cited answer, a quote check and a
  trace record per request.
- Put the engine behind a Gradio page that streams its progress and shows the answer only after the check.
- Test a page from code as several visitors with `gradio_client`, and read what the screen showed update by update.
- Explain why a field the visitor fills, shown or hidden, cannot say who is asking, and take the user from a sign-in.
- Run the service from `app.py` and open it on your route.
- Run Lab 11's release gate on the service's own results, and set it up as a CI job that needs no key.

## Going further: keep the index between starts

`app.py` embeds the corpus every time it starts. `InMemoryVectorStore` can save its documents and vectors to a JSON
file (`dump`) and read them back (`load`) without calling the embedding model. Time both against a fresh build:

In [27]:
from langchain_core.vectorstores import InMemoryVectorStore
from shared.settings import lc_embeddings

saved = ROOT / ".cache" / "capstone" / f"store-{settings['EMBED_MODEL']}.json"
start = time.perf_counter(); app.store.dump(str(saved)); dumped = time.perf_counter() - start
start = time.perf_counter(); loaded = InMemoryVectorStore.load(str(saved), lc_embeddings(settings)); read = time.perf_counter() - start
start = time.perf_counter(); build_store(); built = time.perf_counter() - start
print(f"dump {dumped:.2f} s ({saved.stat().st_size // 1024} KB) | load {read:.2f} s, {len(loaded.store)} documents | build {built:.2f} s")
print("loaded, as a customer:", [doc.id for doc, _ in loaded.similarity_search_with_score(STAFF_QUESTION, k=3, filter=can_read("customer"))])

dump 0.34 s (11210 KB) | load 0.09 s, 281 documents | build 1.29 s


loaded, as a customer: ['H02', 'H01', 'H04']


Saving took 0.34 s and wrote an 11 MB file; loading took 0.09 s with no embedding call, against 1.29 s for a fresh build, and the loaded index gives the customer the same H02, H01, H04. At 281 short documents that saves about a second per start; with a hundred thousand documents, or
several copies of the service, the index belongs in a database every copy reads. The file is named by the embedding
model, because vectors from another model do not fit.

In LangChain a persistent store is a one-line change. With [Qdrant](https://docs.langchain.com/oss/python/integrations/vectorstores/qdrant)
(`pip install langchain-qdrant`), for example:

```python
store = QdrantVectorStore.from_documents(documents, lc_embeddings(settings), path=".cache/qdrant", collection_name="rag")
```

The permission filter does not carry over: a database store takes its own filter syntax, not a Python function. Qdrant takes `models.Filter(must=[models.FieldCondition(key="metadata.access",
match=models.MatchAny(any=USERS[user]))])`, and [PGVector](https://docs.langchain.com/oss/python/integrations/vectorstores/pgvector)
`filter={"access": {"$in": USERS[user]}}`. Rewrite `can_read`, then run Parts C and E again: the gate's access test is
what tells you the new filter still holds.

## Challenge

Use only what this lab showed. A worked solution is in `solved/` for the instructor. Task 1 is the challenge; task 2
is optional. For each task, keep the printed results and your answer to **Explain**.

1. **Read the service's traces.** What did each kind of user cost, and did any answer fail the quote check?

   - Under **Task 1**, read the trace file this notebook wrote (Parts A to E) and print, for each user, the number of
     requests, the mean input tokens, the mean seconds and the answers that failed the check; then the slowest request.

   **Check:** two lines, one per user, and the slowest request. Our worked solution counted 45 customer and 13 agent requests (449 and 323 input tokens on average; one customer answer failed the check), and a slowest request of 1.67 s.

   **Explain:** Part C's customer who said *agent* is in these counts. Under which user, and what would the trace need
   to record to show who really asked?

2. **Optional: stop a right answer from failing the quote check.** The answer to *How many fumbles did Von Miller
   force in Super Bowl 50?* is right, and it often fails the check, because the model shortens its quote with `...`
   and the check compares whole runs of words.

   - Under **Task 2**, answer the question ten times with the answer rules as they are and ten times with one line
     added (`LINE`, written for you), and count the replies that fail the check.

   **Check:** two counts. Our worked solution: 6 of 10 with the rules as they are, 0 of 10 with the line.

   **Explain:** the line changes the answer rules. Which saved files must be made again, and which test run again,
   before this change ships?

**Task 1.** Complete the cell, then run it:

In [28]:
traces = [json.loads(line) for line in TRACES.read_text(encoding="utf-8").splitlines()]
print(len(traces), "trace records")
# TODO: for each user in USERS, take the traces whose "user" is that user and print how many there are,
#       their mean input tokens (trace["usage"]["input"]), their mean seconds and how many failed the quote
#       check (not trace["checked"]); then print the seconds, user and question of the slowest request

58 trace records


**Task 2 (optional).** Complete the cell, then run it:

In [29]:
LINE = " Copy each quote whole, as one run of words from its line; never shorten it with '...'."
miller = "How many fumbles did Von Miller force in Super Bowl 50?"
miller_lines = lc_pipeline.evidence(lc_pipeline.retrieve(store, miller, filter=can_read("customer")))
# TODO: for rules in (RULES, RULES + LINE), answer miller ten times with
#       lc_pipeline.answer(miller, miller_lines, settings, rules=rules), and count the replies that the check in
#       answer_and_trace would fail; print both counts

## Troubleshooting

| Symptom | Likely cause | Fix |
|---|---|---|
| `ModuleNotFoundError: No module named 'gradio'` | the environment was installed without the capstone's packages | uv: `uv sync --group capstone`; pip: `pip install "gradio>=6,<7"` |
| `ModuleNotFoundError: No module named 'gradio_client'` in the notebook, or `AttributeError: module 'gradio' has no attribute 'Request'` from `app.py` | a later `uv sync` without `--group capstone` removed Gradio and left one file behind, so `gradio` imports as an empty folder | `uv sync --group capstone` |
| `ValueError: Could not login to http://127.0.0.1:7860/.` | a wrong password; `app.py` makes new ones at every start | use the passwords from the last start |
| `AppError: The upstream Gradio app has raised an exception ...` in the client, and `KeyError: None` in the server's output | the page was launched without `auth=`, so `request.username` is `None` | launch with `auth=` (`python app.py` does) |
| `OPENAI_API_KEY not found: put it in .env, or set it in your shell or as a Codespaces secret.` when `app.py` starts | no key in the environment or `.env` | add the key as in Lab 00, then start it again |

## Resources

- [Gradio: Interface](https://www.gradio.app/docs/gradio/interface) and
  [Request](https://www.gradio.app/docs/gradio/request): a function as a page, and who sent each request
- [Gradio: sharing your app](https://www.gradio.app/guides/sharing-your-app): `share=True` links, password-protected
  pages and `/logout`
- [Gradio: the Python client](https://www.gradio.app/guides/getting-started-with-the-python-client): `Client`,
  `predict`, `submit` and `view_api`
- [LangChain: `get_usage_metadata_callback`](https://reference.langchain.com/python/langchain-core/callbacks/usage/get_usage_metadata_callback):
  token counts of every call in a block
- [GitHub Actions: quickstart](https://docs.github.com/en/actions/writing-workflows/quickstart): where workflow files go
  and how they run
- [OWASP: LLM02 sensitive information disclosure](https://genai.owasp.org/llmrisk/llm022025-sensitive-information-disclosure/):
  why access control belongs before the model

Back to [the lab list](../README.md).